# Trabalho 1 de Aprendizado de Máquina
## Dataset: Seoul Bike Sharing Demand
#### Integrantes: Felipe Pasinato Rossoni, Renan Augusto da Silva Zen, Stefano Fiorioli Dahmer

O dataset contém informações sobre quantas bicicletas foram alugadas por hora na cidade de Seoul (capital da Coreia do Sul), com informações sobre: Data, Hora, Qtd de Bicicletas alugadas, Temperatura, Chuva, Neve, Umidade, Estação, Feriado, entre outros.

O objetivo do modelo será prever, dadas as condições de um determinado dia/hora, quantas bicicletas serão alugadas.

Atributos do dataset:
Date, Rented Bike Count, Hour, Temperature (°C), Humidity (%), Wind speed (m/s), Visibility (10m), Dew point temperature (°C), Solar Radiation (Mj/m²), Rainfall (mm), Snowfall (cm), Seasons, Holiday, Functioning Day.


In [10]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor


In [11]:
df = pd.read_csv('SeoulBikeData.csv')
print(df.head())
print(df.info())

         Date  Rented Bike Count  Hour  Temperature  Humidity  Wind speed  \
0  01/12/2017                254     0         -5.2        37         2.2   
1  01/12/2017                204     1         -5.5        38         0.8   
2  01/12/2017                173     2         -6.0        39         1.0   
3  01/12/2017                107     3         -6.2        40         0.9   
4  01/12/2017                 78     4         -6.0        36         2.3   

   Visibility  Dew point temperature  Solar Radiation  Rainfall  Snowfall  \
0        2000                  -17.6              0.0       0.0       0.0   
1        2000                  -17.6              0.0       0.0       0.0   
2        2000                  -17.7              0.0       0.0       0.0   
3        2000                  -17.6              0.0       0.0       0.0   
4        2000                  -18.6              0.0       0.0       0.0   

  Seasons     Holiday Functioning Day  
0  Winter  No Holiday             

## Pré-processamento

In [12]:
# deixa so dias que functioning day = Yes no dataset (dias com No sempre tem bike count = 0)
df = df[df['Functioning Day'] == 'Yes'].copy()
df = df.drop('Functioning Day', axis=1)

# separa dia e mes pra ver tendencias
df['Date'] = pd.to_datetime(df['Date'], format='%d/%m/%Y')
df['Month'] = df['Date'].dt.month
df['DayOfWeek'] = df['Date'].dt.dayofweek # 0 = segunda, 6 = domingo
df = df.drop('Date', axis=1)

X = df.drop('Rented Bike Count', axis=1)
y = df['Rented Bike Count']

# separa em treino e teste sem embaralhar (para nao receber informacoes futuras)
X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.2, shuffle=False
)


In [13]:
numeric_features = X_train.select_dtypes(include=["number"]).columns.tolist()
categorical_features = X_train.select_dtypes(exclude=["number"]).columns.tolist()

numeric_transformer = Pipeline(steps=[
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('onehot', OneHotEncoder(drop='first', handle_unknown='ignore')) # evita colunas redundantes
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ])

